# Эмбеддинги e5 (small, base, large) для prompt-guard

Считает эмбеддинги выборок тремя моделями на GPU, чтобы не считать их на ноутбуке.

1. Runtime -> Change runtime type -> **T4 GPU**.
2. Запускайте ячейки сверху вниз.
3. В ячейке загрузки выберите три файла из `data/splits/` проекта: `train.jsonl`, `val.jsonl`, `golden.jsonl`.
4. В конце скачается архив `e5_embeddings.zip`. Распакуйте его в `results/cache/embeddings/` проекта.

Формат результата совпадает с локальным кэшем (префикс `query: `, нормализованные векторы, контрольная сумма файла выборки).

In [ ]:
!pip install -q sentence-transformers
import torch
assert torch.cuda.is_available(), "включите GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
uploaded = files.upload()  # выберите train.jsonl, val.jsonl, golden.jsonl
for name in ("train.jsonl", "val.jsonl", "golden.jsonl"):
    assert name in uploaded, f"не загружен {name}"
print({k: len(v) for k, v in uploaded.items()})

In [ ]:
import gc, hashlib, json, os, time
import numpy as np
from sentence_transformers import SentenceTransformer

MODELS = ["intfloat/multilingual-e5-small", "intfloat/multilingual-e5-base", "intfloat/multilingual-e5-large"]
PREFIX = "query: "
os.makedirs("out", exist_ok=True)

for MODEL in MODELS:
    short = MODEL.split("/")[-1]
    model = SentenceTransformer(MODEL, device="cuda")
    for split in ("train", "val", "golden"):
        raw = open(f"{split}.jsonl", "rb").read()
        texts = [json.loads(line)["text"] for line in raw.decode("utf-8").splitlines() if line.strip()]
        t0 = time.time()
        vecs = model.encode([PREFIX + t for t in texts], batch_size=64, normalize_embeddings=True,
                            show_progress_bar=False, convert_to_numpy=True).astype(np.float32)
        np.save(f"out/{short}__{split}.npy", vecs)
        meta = {"model": MODEL, "prefix": PREFIX, "split": split, "split_sha256": hashlib.sha256(raw).hexdigest(),
                "n": len(texts), "dim": int(vecs.shape[1]), "seconds": round(time.time() - t0, 1)}
        json.dump(meta, open(f"out/{short}__{split}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
        print(short, split, vecs.shape, meta["seconds"], "с", flush=True)
    del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
import shutil
shutil.make_archive("e5_embeddings", "zip", "out")
files.download("e5_embeddings.zip")